# T5 — Preguntas guía y conclusiones finales
**Autor:** Daniel (jefe de grupo)
**Proyecto:** P1 — Análisis exploratorio del Titanic
**Asignatura:** Python para IA — 2º DAM

Esta sección responde a las preguntas guía del enunciado apoyándose en los resultados de T1-T4 y cierra con las conclusiones finales.

In [1]:
import pandas as pd
import numpy as np

# Carga del dataset desde la carpeta Data del repositorio.
# Si no se encuentra (por ejemplo al abrir el notebook en Colab), se descarga
# de la misma fuente original, así que los datos son idénticos.
try:
    df = pd.read_csv("../Data/titanic.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

df.shape

(891, 15)

Aplicamos la base común del contrato (nulos, ver T2a) y recreamos la franja de edad de T3a. Guardamos también una copia de los datos originales para comprobar el efecto de la imputación de edad.

In [2]:
df_original = df.copy()  # datos sin tratar, para la pregunta 5

# --- BASE COMÚN (tratamiento de nulos acordado en el contrato, ver T2a) ---
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df = df.drop(columns="deck")

# Recreamos la franja de edad igual que en T3a
edad = df["age"]
condiciones = [edad < 13, (edad >= 13) & (edad < 18), (edad >= 18) & (edad < 60), edad >= 60]
valores = ["0-12", "13-17", "18-59", "60+"]
df["franja_edad"] = np.select(condiciones, valores, default="Desconocido")

## Pregunta 1: ¿Qué porcentaje de pasajeros sobrevivió en total?

In [3]:
print("Supervivientes:", df["survived"].sum(), "de", len(df))
round(df["survived"].mean() * 100, 2)

Supervivientes: 342 de 891


np.float64(38.38)

**Respuesta:** sobrevivió el 38,38% de los pasajeros (342 de 891).

## Pregunta 2: ¿Cómo varía la tasa de supervivencia según el sexo?

In [4]:
(df.groupby("sex")["survived"].mean() * 100).round(2)

sex
female    74.20
male      18.89
Name: survived, dtype: float64

**Respuesta:** sobrevivió el 74,20% de las mujeres frente al 18,89% de los hombres. Es la variable individual que más diferencia marca en todo el dataset.

## Pregunta 3: ¿Y según la clase del billete (1ª, 2ª, 3ª)?

In [5]:
(df.groupby("pclass")["survived"].mean() * 100).round(2)

pclass
1    62.96
2    47.28
3    24.24
Name: survived, dtype: float64

**Respuesta:** 62,96% en 1ª clase, 47,28% en 2ª y 24,24% en 3ª. La supervivencia baja de forma constante a medida que baja la clase.

## Pregunta 4: ¿Existe relación entre la edad y la probabilidad de supervivencia?

In [6]:
print("Correlación edad-supervivencia (con edades imputadas):", round(df["age"].corr(df["survived"]), 3))
print("Correlación edad-supervivencia (solo edades reales):", round(df_original["age"].corr(df_original["survived"]), 3))

Correlación edad-supervivencia (con edades imputadas): -0.065
Correlación edad-supervivencia (solo edades reales): -0.077


**Respuesta:** la correlación lineal entre edad y supervivencia es débil y negativa (-0,065; -0,077 usando solo las edades reales), casi nula. Pero eso no significa que la edad no influya: agrupando por franjas (pregunta 5) sí aparece una relación, concentrada en los pasajeros más jóvenes y los más mayores.

## Pregunta 5: ¿Qué franja de edad tiene mayor proporción de supervivientes?

In [7]:
# Tasa de supervivencia (media de survived) y número de pasajeros por franja
tabla_edad = df.groupby("franja_edad")["survived"].agg(["mean", "size"])
tabla_edad["mean"] = (tabla_edad["mean"] * 100).round(2)
tabla_edad.columns = ["tasa_supervivencia_%", "num_pasajeros"]
tabla_edad

,tasa_supervivencia_%,num_pasajeros
franja_edad,,
0-12,57.97,69
13-17,47.73,44
18-59,36.44,752
60+,26.92,26


In [8]:
# Comprobación: la misma tabla usando solo los pasajeros con edad real (sin imputar)
reales = df_original.dropna(subset=["age"]).copy()
edad_r = reales["age"]
reales["franja_edad"] = np.select(
    [edad_r < 13, (edad_r >= 13) & (edad_r < 18), (edad_r >= 18) & (edad_r < 60), edad_r >= 60],
    valores, default="Desconocido")
(reales.groupby("franja_edad")["survived"].mean() * 100).round(2)

franja_edad
0-12     57.97
13-17    47.73
18-59    38.61
60+      26.92
Name: survived, dtype: float64

**Respuesta:** la franja 0-12 años tiene la mayor proporción de supervivientes (57,97%), por delante de 13-17 (47,73%), 18-59 (36,44%) y 60+ (26,92%). La supervivencia baja de una franja a la siguiente, lo que es coherente con la prioridad a "mujeres y niños" en los botes. Aun así, la correlación de la pregunta 4 es casi nula porque la diferencia se concentra en los extremos (niños y mayores de 60), que son pocos pasajeros (95 de 891), mientras que la gran mayoría (84%) está en la franja 18-59.

La imputación de la edad solo afecta a la franja 18-59, que es donde caen los 177 pasajeros sin edad: con ellos baja del 38,61% al 36,44%, porque esos pasajeros sobrevivieron menos (29,38%). Las demás franjas no cambian y el orden entre franjas es el mismo.

## Pregunta 6 (opcional): ¿La combinación de sexo y clase cambia mucho la tasa de supervivencia respecto a mirar cada variable por separado?

In [9]:
(df.groupby(["sex", "pclass"])["survived"].mean() * 100).round(2).unstack()

pclass,1,2,3
sex,,,
female,96.81,92.11,50.00
male,36.89,15.74,13.54


**Respuesta:** sí. Por separado, podría parecer que sexo y clase suman sus efectos de forma independiente. Al cruzarlos se ve que, entre las mujeres, la clase apenas cambia la supervivencia en 1ª y 2ª (96,81% y 92,11%) y solo cae con fuerza en 3ª (50,00%). Entre los hombres, la supervivencia es baja en las tres clases (36,89%, 15,74% y 13,54%). Viajar en 1ª clase protegía mucho más a una mujer (+46,8 puntos respecto a 3ª) que a un hombre (+23,4 puntos).

## Conclusiones finales

1. **El sexo es la variable que más determina la supervivencia.** Sobrevivió el 74,20% de las mujeres frente al 18,89% de los hombres, 55 puntos de diferencia, más que cualquier otra variable analizada.

2. **La clase del billete también influye, y de forma escalonada.** La supervivencia baja del 62,96% en 1ª clase al 47,28% en 2ª y al 24,24% en 3ª. La tarifa media en 1ª es más de 6 veces la de 3ª, así que clase y poder adquisitivo van juntos.

3. **Sexo y clase se combinan, no actúan por separado.** Casi todas las mujeres de 1ª y 2ª clase sobrevivieron (más del 92%), pero solo la mitad de las de 3ª. Entre los hombres la supervivencia fue baja en todas las clases. Mirar las dos variables juntas cuenta una historia distinta a mirarlas por separado.

4. **La edad influye, sobre todo en los extremos.** La correlación entre edad y supervivencia es casi nula (-0,065), pero al agrupar por franjas los niños de 0-12 años tienen la mayor tasa de supervivencia (57,97%) y los mayores de 60 la menor (26,92%). Como son pocos pasajeros, el efecto casi no se nota en la correlación global.

5. **Los datos tienen limitaciones.** Cerca del 20% de las edades son estimaciones (mediana imputada), lo que baja algo la tasa de la franja 18-59. Además, faltan columnas identificadoras (nombre, billete), así que no se puede confirmar si las 107 filas repetidas son pasajeros distintos; con los datos disponibles lo más probable es que sí, y por eso no se eliminaron (ver T2b).